In [3]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split

from capstone.dataset import build_dataset, dedup_dataset, select_feature_columns
from capstone.prep import prep_data
from capstone.classic import train_classical_model, classical_predict_proba
from capstone.evaluate import evaluate_model
from capstone.dnn import train_dnn

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
print(tf.reduce_sum(tf.random.normal([1000, 1000])))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
tf.Tensor(665.8134, shape=(), dtype=float32)


## TODO

In [5]:
def prepare_experiment(
    raw: dict[str, pd.DataFrame], 
    stratify_by_source : bool, 
    test_size : float = 0.2, 
    random_state : int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    df = build_dataset(raw)
    df = dedup_dataset(df)

    if stratify_by_source:
        strat_key = df["Label"].astype(str) + "_" + df["Source"]
    else:
        strat_key = df["Label"]
    
    df_train, df_test = train_test_split(df, test_size=test_size, stratify=strat_key, random_state=random_state)

    exclude_cols = ["Source", "Subject", "Body", "Label", "text"]
    candidate_cols = [c for c in df.columns if c not in exclude_cols]
    feature_cols = select_feature_columns(df_train, candidate_cols)

    return df_train, df_test, feature_cols



In [6]:
# Download the consoldiated spam data set from kaggle; the kaggle API handles local caching
path = kagglehub.dataset_download("nitishabharathi/email-spam-dataset")

raw_data_sets = { 
    'Enron': pd.read_csv(f"{path}/enronSpamSubset.csv"),
    'Spam Assassin': pd.read_csv(f"{path}/completeSpamAssassin.csv"),
    'LingSpam': pd.read_csv(f"{path}/lingSpam.csv")
}

In [7]:
enron_train, enron_test, enron_features = prepare_experiment(
    { "Enron" : raw_data_sets["Enron"] }, stratify_by_source=False
)
combined_train, combined_test, combined_features = prepare_experiment(
    raw_data_sets, stratify_by_source=True
)

# comfortably above the largest genuine outlier (23,343), well below the corrupted row (3,527,577)
CORRUPTION_THRESHOLD = 100_000  

token_counts = combined_train["text"].str.split().str.len()
combined_train = combined_train[token_counts <= CORRUPTION_THRESHOLD]


## TODO

In [ ]:
%%time
enron_grid = train_classical_model(enron_train, feature_cols=enron_features, verbose=1)
print("Enron only best parameters: ", enron_grid.best_params_)

Enron only best parameters:  {'clf__C': 100, 'features__tfidf__max_features': 20000, 'features__tfidf__ngram_range': (1, 1)}
CPU times: user 7min 42s, sys: 1.78 s, total: 7min 44s
Wall time: 1min 15s


In [ ]:
%%time
combined_grid = train_classical_model(combined_train, feature_cols=combined_features, verbose=1))
print("Combined best parameters: ", combined_grid.best_params_)

Combined best parameters:  {'clf__C': 100, 'features__tfidf__max_features': None, 'features__tfidf__ngram_range': (1, 1)}
CPU times: user 8min 37s, sys: 5.95 s, total: 8min 43s
Wall time: 2min 52s


### TODO

In [10]:
df_full = pd.concat([combined_train, combined_test], ignore_index=True)
df_full = df_full[df_full["Source"] != "Enron"]

enron_predict_proba = classical_predict_proba(enron_grid.best_estimator_)
combined_predict_proba = classical_predict_proba(combined_grid.best_estimator_)

results = { 
    "Enron only" : evaluate_model(enron_predict_proba, enron_test, "Label"),
    "Enron only (isolated test data)" : evaluate_model(enron_predict_proba, df_full, "Label"),
    "Combined": evaluate_model(combined_predict_proba, combined_test, "Label")
}

In [11]:
for source_name, source_df in combined_test.groupby("Source"):
    results[f"Combined ({source_name})"] = evaluate_model(combined_predict_proba, source_df, "Label")

for name, metrics in results.items():
    print(f"{name}: precision={metrics['precision']:.3f} recall={metrics['recall']:.3f} "
          f"f1={metrics['f1']:.3f} roc_auc={metrics['roc_auc']:.3f}")

Enron only: precision=0.986 recall=0.989 f1=0.988 roc_auc=0.998
Enron only (isolated test data): precision=0.274 recall=0.989 f1=0.429 roc_auc=0.595
Combined: precision=0.980 recall=0.985 f1=0.983 roc_auc=0.996
Combined (Enron): precision=0.982 recall=0.986 f1=0.984 roc_auc=0.996
Combined (LingSpam): precision=0.933 recall=0.976 f1=0.954 roc_auc=0.988
Combined (Spam Assassin): precision=0.989 recall=0.982 f1=0.985 roc_auc=0.995


### DNN

In [14]:
def dnn_predict_proba(model, feature_cols, verbose : int = 1):
    def predict(df):
        text = df["text"].to_numpy(dtype=object)
        features = df[feature_cols].to_numpy(dtype=np.float64)
        predictions = model.predict({
            "text": text,
            "engineered_features": features
        }, verbose=verbose)
        return predictions.ravel()
    return predict

In [12]:
%%time
enron_dnn = train_dnn(enron_train, enron_features)

Epoch 1/15


/home/masimms/code/aiml-course-capstone/.venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'lstm' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


218/218 ━━━━━━━━━━━━━━━━━━━━ 338s 2s/step - accuracy: 0.9041 - auc: 0.9628 - loss: 0.2491 - precision: 0.9121 - recall: 0.8888 - val_accuracy: 0.9110 - val_auc: 0.9843 - val_loss: 0.2364 - val_precision: 0.8556 - val_recall: 1.0000
Epoch 2/15
218/218 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - accuracy: 0.9842 - auc: 0.9959 - loss: 0.0617 - precision: 0.9746 - recall: 0.9935 - val_accuracy: 0.9871 - val_auc: 0.9971 - val_loss: 0.0530 - val_precision: 0.9784 - val_recall: 0.9976
Epoch 3/15
218/218 ━━━━━━━━━━━━━━━━━━━━ 336s 2s/step - accuracy: 0.9924 - auc: 0.9990 - loss: 0.0278 - precision: 0.9906 - recall: 0.9938 - val_accuracy: 0.9652 - val_auc: 0.9911 - val_loss: 0.1151 - val_precision: 0.9401 - val_recall: 0.9976
Epoch 4/15
218/218 ━━━━━━━━━━━━━━━━━━━━ 336s 2s/step - accuracy: 0.9934 - auc: 0.9986 - loss: 0.0252 - precision: 0.9892 - recall: 0.9974 - val_accuracy: 0.9794 - val_auc: 0.9937 - val_loss: 0.0727 - val_precision: 0.9735 - val_recall: 0.9878
Epoch 5/15
218/218 ━━━━━━━━━━━━━━━━━━━━

In [13]:
%%time
combined_dnn = train_dnn(combined_train, combined_features)

Epoch 1/15


/home/masimms/code/aiml-course-capstone/.venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'lstm_1' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


396/396 ━━━━━━━━━━━━━━━━━━━━ 594s 1s/step - accuracy: 0.9208 - auc: 0.9743 - loss: 0.2056 - precision: 0.8744 - recall: 0.9196 - val_accuracy: 0.9616 - val_auc: 0.9921 - val_loss: 0.1145 - val_precision: 0.9838 - val_recall: 0.9137
Epoch 2/15
396/396 ━━━━━━━━━━━━━━━━━━━━ 589s 1s/step - accuracy: 0.9822 - auc: 0.9971 - loss: 0.0578 - precision: 0.9714 - recall: 0.9811 - val_accuracy: 0.9787 - val_auc: 0.9955 - val_loss: 0.0744 - val_precision: 0.9581 - val_recall: 0.9869
Epoch 3/15
396/396 ━━━━━━━━━━━━━━━━━━━━ 586s 1s/step - accuracy: 0.9892 - auc: 0.9984 - loss: 0.0365 - precision: 0.9797 - recall: 0.9915 - val_accuracy: 0.9616 - val_auc: 0.9907 - val_loss: 0.1353 - val_precision: 0.9209 - val_recall: 0.9831
Epoch 4/15
396/396 ━━━━━━━━━━━━━━━━━━━━ 587s 1s/step - accuracy: 0.9944 - auc: 0.9995 - loss: 0.0189 - precision: 0.9891 - recall: 0.9960 - val_accuracy: 0.9765 - val_auc: 0.9935 - val_loss: 0.0849 - val_precision: 0.9753 - val_recall: 0.9625
Epoch 5/15
396/396 ━━━━━━━━━━━━━━━━━━━━

In [17]:
df_clean[df_clean["Source"] != "Enron"]

,Source,Subject,Body,Label,subject_length,subject_word_count,subject_upper_ratio,subject_digit_ratio,subject_has_http_url,subject_has_web_url,...,body_rbracket_count,body_caret_count,body_underscore_count,body_backtick_count,body_lbrace_count,body_pipe_count,body_rbrace_count,body_tilde_count,has_subject,text
2,LingSpam,"summary : "" grasshopper mind ""","\n short answer : "" grasshopper mind "" is bri...",0,30,6,0.0,0.0,False,False,...,0,0,16,0,0,0,0,0,True,"summary : "" grasshopper mind "" \n short answe..."
3,Spam Assassin,,New amazing incest show on Hot-Babies-Live.Com...,1,0,0,0.0,0.0,False,False,...,0,0,0,0,0,0,0,0,False,New amazing incest show on Hot-Babies-Live.Co...
7,Spam Assassin,,http://www.hughes-family.org/bugzilla/show_bug...,0,0,0,0.0,0.0,False,False,...,0,0,48,0,0,0,0,0,False,http://www.hughes-family.org/bugzilla/show_bu...
8,Spam Assassin,,"On Mon, 2002-09-30 at 09:20, Owen Byrne wrote:...",0,0,0,0.0,0.0,False,False,...,0,0,0,0,0,0,0,0,False,"On Mon, 2002-09-30 at 09:20, Owen Byrne wrote..."
10,LingSpam,keeps america online online,\n this is not spam ; you are receiving this ...,1,27,4,0.0,0.0,False,False,...,0,0,0,0,0,0,0,0,True,keeps america online online \n this is not sp...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17555,Spam Assassin,,\nDid you know 4 of the country's 10 richest p...,1,0,0,0.0,0.0,False,False,...,0,0,0,0,0,0,0,0,False,\nDid you know 4 of the country's 10 richest ...
17558,Spam Assassin,,> From: Stephen D. Williams [mailto:swilliams@...,0,0,0,0.0,0.0,False,False,...,1,0,0,0,0,0,0,0,False,> From: Stephen D. Williams [mailto:swilliams...
17560,Spam Assassin,,"Spray a bit of wd40 around the screw, and leav...",0,0,0,0.0,0.0,False,False,...,0,0,0,0,0,0,0,1,False,"Spray a bit of wd40 around the screw, and lea..."
17563,Spam Assassin,,John P. Looney wrote:\n> I've two directories...,0,0,0,0.0,0.0,False,False,...,0,0,0,0,0,0,0,0,False,John P. Looney wrote:\n> I've two directorie...


In [19]:
df_clean = pd.concat([combined_train, combined_test], ignore_index=True)
df_clean = df_clean[df_clean["Source"] != "Enron"]

enron_dnn_predict_proba = dnn_predict_proba(enron_dnn, enron_features)
combined_dnn_predict_proba = dnn_predict_proba(combined_dnn, combined_features)

dnn_results = {
    "Enron-only (in-distribution)": evaluate_model(enron_dnn_predict_proba, enron_test, "Label"),
    "Enron-only (out-of-distribution)": evaluate_model(enron_dnn_predict_proba, df_clean, "Label"),
    "Combined (overall)": evaluate_model(combined_dnn_predict_proba, combined_test, "Label"),
}

for source_name, source_df in combined_test.groupby("Source"):
    dnn_results[f"Combined ({source_name})"] = evaluate_model(combined_dnn_predict_proba, source_df, "Label")

for name, metrics in dnn_results.items():
    print(f"{name}: precision={metrics['precision']:.3f} recall={metrics['recall']:.3f} "
          f"f1={metrics['f1']:.3f} roc_auc={metrics['roc_auc']:.3f}")

61/61 ━━━━━━━━━━━━━━━━━━━━ 29s 481ms/step
247/247 ━━━━━━━━━━━━━━━━━━━━ 118s 478ms/step


/home/masimms/code/aiml-course-capstone/.venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'lstm_1' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


110/110 ━━━━━━━━━━━━━━━━━━━━ 52s 476ms/step
61/61 ━━━━━━━━━━━━━━━━━━━━ 29s 481ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 484ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 17s 490ms/step
Enron-only (in-distribution): precision=0.967 recall=0.995 f1=0.981 roc_auc=0.999
Enron-only (out-of-distribution): precision=0.245 recall=0.988 f1=0.392 roc_auc=0.744
Combined (overall): precision=0.961 recall=0.979 f1=0.970 roc_auc=0.997
Combined (Enron): precision=0.975 recall=0.980 f1=0.977 roc_auc=0.996
Combined (LingSpam): precision=0.882 recall=0.965 f1=0.921 roc_auc=0.992
Combined (Spam Assassin): precision=0.941 recall=0.982 f1=0.961 roc_auc=0.998
